Strategic Workforce Attrition Data Analysis

Project Overview

This notebook performs end-to-end data analysis on employee attrition across five related datasets:

| Dataset | Variable | Description |
|---|---|---|
| Employee.csv | `df` | Core employee records |
| Department.csv | `dt` | Department details and budgets |
| Manager.csv | `mg` | Manager profiles |
| Salary.csv | `sal` | Salary history |
| Attrition_date.csv | `at` | Employee exit events |

Notebook Structure
1. Imports & Setup
2. Data Loading
3. Data Cleaning — Employee Table
4. Data Cleaning — Department Table
5. Data Cleaning — Manager Table
6. Data Cleaning — Salary Table
7. Data Cleaning — Attrition Table
8. Exploratory Data Analysis (EDA)
9. Data Export
10. Visualizations


1. Imports & Setup

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from word2number import w2n

 2. Data Loading

2.1 Employee Table

In [ ]:
# Load the Employee dataset
df = pd.read_csv('D:\Attrition Analysis\Raw_data_set\Employee.csv')
df

2.2 Department Table

In [ ]:
# Load the Department dataset
dt = pd.read_csv('D:\\Attrition Analysis\\Raw_data_set\\Department.csv')
dt

2.3 Manager Table

In [ ]:
# Load the Manager dataset
mg = pd.read_csv('D:\\Attrition Analysis\\Raw_data_set\\Manager.csv')
mg

 2.4 Salary Table

In [ ]:
# Load the Salary dataset
sal = pd.read_csv('D:\\Attrition Analysis\\Raw_data_set\\Salary.csv')
sal

 Attrition Table

In [ ]:
# Load the Attrition (Employee Exit) dataset
at = pd.read_csv('D:\\Attrition Analysis\\Raw_data_set\\Attrition_date.csv')
at

3. Data Cleaning — Employee Table (`df`)

3.1 Initial Inspection

In [ ]:
# Display raw column names before cleaning
df.columns

In [ ]:
display(df.info())      # Review dataset structure, data types, and missing values
display(df.describe())  # Generate descriptive statistics
display(df.head(10))    # Preview dataset

3.2 Column Name Cleanup

In [ ]:
# Strip leading and trailing whitespace from all column names
df.columns = df.columns.str.strip()

 3.3 employee_id — Duplicates & Missing Values

In [ ]:
# Check missing values in employee_id
df['employee_id'].isnull().sum()

In [ ]:
df['employee_id'].duplicated().unique().sum()  # Check duplicate employee IDs

In [ ]:
# Remove duplicate employee records based on employee_id
print(f"Original rows: {len(df)}")
df = df.drop_duplicates(subset='employee_id', keep='first')
print(f"Rows after deleteduplicate: {len(df)}")

 3.4 Name Columns — Whitespace & Title Case

In [ ]:
# Remove leading and trailing spaces, and apply title case to first_name
df['first_name'] = df['first_name'].str.strip().str.title()
df['first_name']

In [ ]:
# Remove leading and trailing spaces, and apply title case to last_name
df['last_name'] = df['last_name'].str.strip().str.title()
df['last_name']

3.5 gender — Missing Values & Standardization

In [ ]:
# Check missing values in gender
df['gender'].isnull().sum()

# Review gender categories
df['gender'].value_counts()

In [ ]:
df['gender'].unique()  # Check unique values in gender column

In [ ]:
# Standardize gender values (remove spaces, normalize case)
df['gender'] = df['gender'].str.strip().str.lower().str.capitalize()
gender_map = {          # Map inconsistent gender labels to standard categories
    'Female': 'Female',
    'F': 'Female',
    'Woman': 'Female',
    'Male': 'Male',
    'M': 'Male',
    'Man': 'Male'
    }
df['gender'] = df['gender'].map(gender_map).fillna('Unknown')
print(df['gender'].unique())

3.6 hire_date — Format & Datetime Conversion

In [ ]:
# Preview raw hire_date values
df['hire_date']

In [ ]:
display(df['hire_date'].unique())       # Check unique hire dates
display(df['hire_date'].nunique())      # Count unique hire dates
display(df['hire_date'].value_counts()) # Frequency of hire dates

In [ ]:
# Convert hire_date to datetime format
df['hire_date'] = pd.to_datetime(df['hire_date'], errors='coerce')
df['hire_date'].head()

In [ ]:
# Verify datetime conversion and check for parsing failures
display(df['hire_date'].dtype)
display(df['hire_date'].isnull().sum())

3.7 salary — Cleaning & Imputation

In [ ]:
df['salary'].unique()  # Check unique salary values

In [ ]:
df.loc[df['salary'] == 'zero']  # Identify problematic values like 'zero'

In [ ]:
# Inspect the specific employee record with a problematic salary
df.loc[df['employee_id'] == 2552]

In [ ]:
# Custom function to parse salary values in various formats (numeric, word, currency symbols, 'k' notation)
def clean_salary(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()

    # Remove currency symbols and commas
    x = x.replace('$', '').replace('₹', '').replace(',', '')

    # Handle 'k' (e.g., 85k, sixty k)
    if 'k' in x:
        # Remove 'k' and try to convert numeric part
        num_part = x.replace('k', '').strip()
        try:
            # If numeric, e.g., '85'
            return float(num_part) * 1000
        except:
            # If word like 'sixty', use w2n
            try:
                return w2n.word_to_num(num_part) * 1000
            except:
                return np.nan

    # Handle 'zero'
    if x == 'zero' or x == 0 or x == '0':
        return np.nan

    # Try word2number for phrases like 'fifty two thousand'
    try:
        return float(w2n.word_to_num(x))
    except:
        pass

    # Try direct float conversion
    try:
        return float(x)
    except:
        return np.nan

# Apply the cleaning function to the salary column
df['salary'] = df['salary'].apply(clean_salary)

# Fill missing salary with median per job role
df['salary'] = df.groupby('job_role')['salary'].transform(lambda x: x.fillna(x.median()))

# Ensure salary is stored as float
df['salary'] = df['salary'].astype(float)

print(df['salary'].head())

In [ ]:
# Verify salary values after cleaning
df['salary'].unique()

In [ ]:
# Confirm the previously problematic employee record is now fixed
df.loc[df['employee_id'] == 2552]

 3.8 dept_id — Invalid Value Handling

In [ ]:
# Check unique department IDs in employee table
df['dept_id'].unique()

In [ ]:
# Replace invalid department ID (99) with NaN
df['dept_id'] = df['dept_id'].replace(99, np.nan)

In [ ]:
# Check distribution after cleaning
df['dept_id'].value_counts()

In [ ]:
# Check missing values in dept_id
df['dept_id'].isnull().sum()

 3.9 manager_id — Invalid Value Handling

In [ ]:
# Check unique manager IDs in employee table
df['manager_id'].unique()

In [ ]:
# Replace invalid manager ID (9999) with NaN
df['manager_id'] = df['manager_id'].replace(9999, np.nan)

 3.10 job_role — Whitespace Cleanup

In [ ]:
df['job_role'].unique()  # Check unique job roles in the dataset

In [ ]:
# Remove leading and trailing spaces from job_role values
df['job_role'] = df['job_role'].str.strip()

 3.11 overtime_hours_per_month — Negative Values & Imputation

In [ ]:
# Check unique values in overtime_hours_per_month
display(df['overtime_hours_per_month'].unique())

# Check frequency distribution
display(df['overtime_hours_per_month'].value_counts())

In [ ]:
# Replace invalid negative values with NaN
df.loc[df['overtime_hours_per_month'] < 0, 'overtime_hours_per_month'] = np.nan

In [ ]:
# Fill missing values with median overtime hours grouped by job_role
df['overtime_hours_per_month'] = df.groupby('job_role')['overtime_hours_per_month'].transform(
    lambda x: x.fillna(x.median()))

 3.12 performance_rating — Missing Value Imputation

In [ ]:
# Check unique performance ratings
display(df['performance_rating'].unique())

# Check frequency distribution
display(df['performance_rating'].value_counts())

In [ ]:
# Fill missing performance_rating with median per job role
df['performance_rating'] = df.groupby('job_role')['performance_rating'].transform(
    lambda x: x.fillna(x.median()))
df['performance_rating']

 3.13 attrition — Review Distribution

In [ ]:
# Check unique values in attrition column
display(df['attrition'].unique())

# Check distribution of attrition values
display(df['attrition'].value_counts())

 3.14 Final Employee Table Review

In [ ]:
# Final overview of the cleaned employee table
df.info()

In [ ]:
# Preview salary values by job role after cleaning
df[['job_role', 'salary']].sort_values('job_role').reset_index(drop=True)

4. Data Cleaning — Department Table (`dt`)

4.1 Initial Inspection

In [ ]:
# Display raw column names before cleaning
dt.columns

In [ ]:
# Dataset overview
display(dt.info())

# Summary statistics
display(dt.describe())

# Preview dataset
display(dt.head())

4.2 Column Name Cleanup

In [ ]:
# Strip leading and trailing whitespace from all column names
dt.columns = dt.columns.str.strip()

 4.3 dept_id — Duplicates Check

In [ ]:
# Count unique department IDs
display(dt['dept_id'].nunique())

# Check duplicate department IDs
display(dt['dept_id'].duplicated().sum())

4.4 dept_name — Whitespace Cleanup

In [ ]:
# Remove leading and trailing spaces from department names
dt['dept_name'] = dt['dept_name'].str.strip()
dt['dept_name']

 4.5 region — Whitespace Cleanup

In [ ]:
# Inspect unique region values before cleaning
dt['region'].unique()

In [ ]:
# Remove leading and trailing spaces from region values
dt['region'] = dt['region'].str.strip()
dt

4.6 budget — Missing Values & Outlier Detection

In [ ]:
# Check for missing values in budget
display(dt['budget'].isnull().sum())

# Review summary statistics
display(dt['budget'].describe())

In [ ]:
# Replace invalid (zero or negative) budget values with NaN
dt.loc[dt['budget'] <= 0, 'budget'] = np.nan

In [ ]:
# Detect outliers using the IQR method
q1 = dt['budget'].quantile(.25)
print(f'q1={q1}')
q3 = dt['budget'].quantile(.75)
print(f'q3={q3}')
iqr = q3 - q1
print(f'IQR={iqr}')
lower = q1 - 1.5 * iqr
print(f'lower_bound={lower}')
upper = q3 + 1.5 * iqr
print(f'upper_bound={upper}')

outliers = (dt['budget'] < lower) | (dt['budget'] > upper)
print(f'outliers={outliers.sum()}')

4.7 Exploratory Questions — Department Table

Q1. Which department has the highest budget?

In [ ]:
dt.sort_values('budget', ascending=False).head()

The IT Infrastructure department has the highest budget (4,860,029), indicating significant investment in technology and operational support.

Q2. What is the regional distribution of departments?

In [ ]:
dt['region'].value_counts()

Most departments are located in the East region (5 departments), followed by Remote operations (4 departments).

Q3. What is the average departmental budget?

In [ ]:
dt['budget'].mean().round()

The average departmental budget is approximately 2,469,601.0

**Department Budget Insight**

Department budgets are unevenly distributed, with IT Infrastructure receiving the highest allocation, indicating priority investment in technical operations.

5. Data Cleaning — Manager Table (`mg`)

 5.1 Initial Inspection

In [ ]:
# Display raw column names before cleaning
mg.columns

In [ ]:
# Strip leading and trailing whitespace from all column names
mg.columns = mg.columns.str.strip()

In [ ]:
# Review dataset structure, data types, and missing values
display(mg.info())

# Generate descriptive statistics
display(mg.describe())

 5.2 manager_id — Missing Values & Duplicates

In [ ]:
# Check for missing values in manager_id
display(mg['manager_id'].isnull().sum())

# Review unique manager IDs
display(mg['manager_id'].nunique())

In [ ]:
# Check duplicate manager IDs
mg['manager_id'].duplicated().sum()

In [ ]:
# Remove duplicate manager records based on manager_id
print(f"Original rows: {len(mg)}")
mg = mg.drop_duplicates(subset='manager_id', keep='first')
print(f"Rows after deleteduplicate: {len(mg)}")

 5.3 manager_name — Whitespace Cleanup

In [ ]:
# Remove leading and trailing spaces from manager names
mg['manager_name'] = mg['manager_name'].str.strip()
mg['manager_name']

5.4 hire_date — Format & Datetime Conversion

In [ ]:
# Inspect hire_date values before conversion
display(mg['hire_date'].unique())
display(mg['hire_date'].nunique())
display(mg['hire_date'].value_counts())

In [ ]:
# Convert hire_date to datetime format
mg['hire_date'] = pd.to_datetime(mg['hire_date'], errors='coerce')
mg['hire_date']

In [ ]:
# Verify data type after conversion
mg['hire_date'].dtype

5.5 performance_rating — Invalid Values & Imputation

In [ ]:
# Inspect performance_rating values and distribution
display(mg['performance_rating'].unique())
display(mg['performance_rating'].isnull().sum())  # Check missing values
display(mg['performance_rating'].value_counts())

In [ ]:
# Review rating distribution in sorted order
mg['performance_rating'].value_counts().sort_index()

In [ ]:
# Replace negative performance ratings with NaN (invalid entries)
mg.loc[mg['performance_rating'] < 0, 'performance_rating'] = np.nan

In [ ]:
# Fill missing performance ratings using the overall median
mg['performance_rating'] = mg['performance_rating'].fillna(mg['performance_rating'].median())
mg['performance_rating']

5.6 dept_id — Referential Integrity Check

In [ ]:
# Check missing values in dept_id
display(mg['dept_id'].isnull().sum())

# Review department IDs
display(mg['dept_id'].unique())

In [ ]:
# Verify that all manager dept_ids exist in the department table (referential integrity check)
set(mg['dept_id']) - set(dt['dept_id'])

 5.7 Final Manager Table Review

In [ ]:
# Review performance rating summary statistics after cleaning
mg['performance_rating'].describe()

In [ ]:
# Final overview of the cleaned manager table
mg.info()

6. Data Cleaning — Salary Table (`sal`)

 6.1 Initial Inspection

In [ ]:
# Check dataset structure, data types, and missing values
display(sal.info())

# Generate descriptive statistics
display(sal.describe())

# Preview first few rows
display(sal.head())

In [ ]:
sal.columns  # Show all columns

 6.2 Column Name Cleanup

In [ ]:
# Remove leading and trailing spaces from column names
sal.columns = sal.columns.str.strip()

6.3 salary_id — Missing Values & Duplicates

In [ ]:
sal['salary_id'].isna().sum()    # Check for missing values in salary_id

In [ ]:
sal['salary_id'].unique()    # Review unique salary IDs

In [ ]:
sal['salary_id'].duplicated().sum()    # Check for duplicate salary IDs

In [ ]:
# Remove duplicate records based on salary_id and keep the first occurrence
print(f"Original rows: {len(sal)}")
sal = sal.drop_duplicates(subset='salary_id', keep='first')
print(f"Rows after deleteduplicate: {len(sal)}")

In [ ]:
sal['salary_id'].duplicated().sum()  # Verify that no duplicate salary IDs remain

 6.4 effective_date — Missing Values & Datetime Conversion

In [ ]:
display(sal['effective_date'].nunique())    # Check the number of unique effective dates
display(sal['effective_date'].isnull().sum())  # Check missing values in effective_date

In [ ]:
# Investigate employees with missing effective dates
missing_emp = sal.loc[sal['effective_date'].isna(), 'employee_id']
sal[sal['employee_id'].isin(missing_emp)].sort_values(
    ['employee_id', 'effective_date'])

In [ ]:
# Convert effective_date from object/string to datetime format
sal['effective_date'] = pd.to_datetime(sal['effective_date'], errors='coerce')
sal['effective_date'].dtype

6.5 monthly_salary — Invalid Values & Outlier Detection

In [ ]:
# Check missing values
display(sal['monthly_salary'].isnull().sum())

# Check salary statistics
display(sal['monthly_salary'].describe())

In [ ]:
# Replace invalid salary values (0 or negative) with NaN
sal.loc[sal['monthly_salary'] <= 0, 'monthly_salary'] = np.nan

In [ ]:
sal['monthly_salary'] = sal['monthly_salary'].round(2)  # Round salary values to 2 decimal places

In [ ]:
# Calculate Q1, Q3 and IQR for monthly_salary
q1 = sal['monthly_salary'].quantile(0.25)
print(f"Q1 (25th Percentile): {q1}")

q3 = sal['monthly_salary'].quantile(0.75)
print(f"Q3 (75th Percentile): {q3}")

iqr = q3 - q1
print(f"IQR: {iqr}")

lower = q1 - 1.5 * iqr
print(f"Lower Bound: {lower}")

upper = q3 + 1.5 * iqr
print(f"Upper Bound: {upper}")

In [ ]:
# Identify and count salary outliers beyond the IQR bounds
outliers = sal[
    (sal['monthly_salary'] < lower) |
    (sal['monthly_salary'] > upper)
]

print(f"Number of Outliers: {len(outliers)}")

 6.6 reason — Missing Values & Whitespace Cleanup

In [ ]:
# Check for missing values in reason column
sal['reason'].isnull().sum()

In [ ]:
display(sal['reason'].unique())        # Review unique salary change reasons
display(sal['reason'].value_counts())  # Check frequency distribution of salary change reasons

In [ ]:
sal['reason'] = sal['reason'].str.strip()  # Remove leading and trailing spaces from reason values

 7. Data Cleaning — Attrition Table (`at`)

7.1 Initial Inspection

In [ ]:
display(at.info())      # Check dataset structure, data types, and missing values
display(at.describe())  # Generate descriptive statistics
display(at.head())      # Preview first few rows

In [ ]:
at.columns  # Show all columns

7.2 Column Name Cleanup

In [ ]:
# Remove leading and trailing spaces from column names
at.columns = at.columns.str.strip()

# Display cleaned column names
print(at.columns.tolist())

7.3 event_id — Duplicates Check & Removal

In [ ]:
at['event_id'].duplicated().value_counts()  # Check for duplicate event IDs

In [ ]:
# Remove duplicate records based on event_id and keep the first occurrence
at = at.drop_duplicates(subset='event_id', keep='first')

In [ ]:
# Verify that no duplicate event IDs remain
at['event_id'].duplicated().sum()

 7.4 employee_id — Referential Integrity & Repeated Records

In [ ]:
# Verify employee IDs exist in employee table (referential integrity check)
set(at['employee_id']) - set(df['employee_id'])

In [ ]:
# Review all unique employee IDs and their frequency in the attrition table
at['employee_id'].unique()
display(at['employee_id'].value_counts())

In [ ]:
# Check for repeated employee IDs (employees with multiple exit events)
at['employee_id'].value_counts()[at['employee_id'].value_counts() > 1]

In [ ]:
# Investigate repeated employee IDs in detail
at[at['employee_id'].duplicated(keep=False)].sort_values('employee_id')

 7.5 attrition_date — Datetime Conversion & Date Range

In [ ]:
# Convert attrition_date column to datetime format
at['attrition_date'] = pd.to_datetime(at['attrition_date'], errors='coerce')
at['attrition_date']

In [ ]:
# Review date range of attrition events
at['attrition_date'].min(), at['attrition_date'].max()

 7.6 attrition_reason — Inspection & Whitespace Cleanup

In [ ]:
# Check unique attrition reasons
at['attrition_reason'].unique()

In [ ]:
# Check frequency distribution of attrition reasons
display(at['attrition_reason'].value_counts())

In [ ]:
# Remove leading and trailing spaces from attrition_reason values
at['attrition_reason'] = at['attrition_reason'].str.strip()
at['attrition_reason']

7.7 exit_interview_score — Invalid Values & Imputation

In [ ]:
# Check unique values in exit_interview_score
at['exit_interview_score'].unique()

In [ ]:
# Count missing values in exit_interview_score
at.loc[at['exit_interview_score'].isnull()].shape[0]

In [ ]:
# Review score distribution
at['exit_interview_score'].value_counts().sort_index()

In [ ]:
# Convert negative exit scores to NaN (invalid entries)
at.loc[at['exit_interview_score'] < 0, 'exit_interview_score'] = np.nan
# Impute missing values with the overall median
at['exit_interview_score'] = at['exit_interview_score'].fillna(at['exit_interview_score'].median())

In [ ]:
# Verify no missing values remain in exit_interview_score
at['exit_interview_score'].isnull().sum()

7.8 Final Attrition Table Review

In [ ]:
# Check missing values
display(at.isnull().sum())

# Check duplicate rows
display(at.duplicated().sum())

# Check data types
at.dtypes

# Verify no duplicate event IDs remain
display(at['event_id'].duplicated().sum())

 8. Exploratory Data Analysis (EDA)

 8.1 Attrition Table — Key Questions

Q1. What is the most common attrition reason?

In [ ]:
at['attrition_reason'].value_counts()

The most common reason for employee attrition was Retirement.

Q2. What is the average exit interview score?

In [ ]:
at['exit_interview_score'].mean().round()

The average exit interview score was 3.2 out of 5.

Q3. What is the average exit score by attrition reason?

In [ ]:
at.groupby('attrition_reason')['exit_interview_score'].mean()

Employees leaving due to Career Change had the lowest satisfaction scores. The second and third most impactful reasons were Relocation and Management Issues.

Q4. What is the attrition trend over time?

In [ ]:
at['attrition_year'] = at['attrition_date'].dt.year
at['attrition_year'].value_counts().sort_index()

Attrition peaked in 2024.

Q5. What is the distribution of exit interview scores?

In [ ]:
at['exit_interview_score'].value_counts().sort_index()

The majority of departing employees gave an exit interview score of 3 (67 employees), indicating a generally neutral sentiment toward the organization at the time of leaving.

8.2 Summary Insights

**Employee Insights:**

- Attrition trend increased over time, indicating rising employee turnover.
- Performance ratings are mostly in the mid-to-high range, showing generally good employee performance.
- Salary varies widely, showing different pay levels across roles.
- Overtime hours differ across employees, indicating workload imbalance.
- Gender distribution is now clean and consistent after standardization.

**Department Insights:**

- IT Infrastructure has the highest budget, indicating priority investment.
- Budget allocation varies significantly across departments.
- Most departments are concentrated in East and Remote regions.

**Data Quality Insights:**

- Invalid department ID (99) and manager ID (9999) were found and corrected.
- Referential integrity between employee, department, and manager tables was ensured.


9. Data Export

Export all cleaned datasets to CSV files for downstream use.

In [ ]:
# Export cleaned employee data
df.to_csv('clean_employee.csv', index=False)

In [ ]:
# Export cleaned salary data
sal.to_csv('clean_salary.csv', index=False)

In [ ]:
# Export cleaned department data
dt.to_csv('clean_department.csv', index=False)

In [ ]:
# Export cleaned manager data
mg.to_csv('clean_manager.csv', index=False)

In [ ]:
# Export cleaned attrition data
at.to_csv('clean_attrition.csv', index=False)

 10. Visualizations

 Q1. How many employees are there in total?

In [ ]:
print(f"Total Number of Employees :{df['employee_id'].nunique()}")

 Q2. Gender Distribution

In [ ]:
gender_count = df['gender'].value_counts()
plt.figure(figsize=(4, 4))

gender_count.plot(kind='pie', colors=["#4ABFCE", "#BDE34C"], autopct='%1.1f%%',
    startangle=90, wedgeprops={'edgecolor': 'white'})
plt.title('Gender Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Insight: The workforce is distributed across Male and Female employees with a relatively balanced representation.

Q3. How many employees are in each job role?

In [ ]:
job_role = df['job_role'].value_counts()
plt.figure(figsize=(5, 5))
job_role.plot(kind='bar', color=["#4E8CF0", "#3EFC6A", "#C5353A", "#FD4524", "#AE42C4", '#3CB371', "#B8DAFF"])
plt.title('Employees per job role', fontweight='bold', fontsize=12, color="#92A21C")
plt.xlabel('Job-Title', fontsize=12)
plt.ylabel('Number of Employees', fontsize=12)
plt.xticks(rotation=45, fontsize=7, fontweight='550')
plt.yticks(fontsize=10, fontweight='550')
for i, v in enumerate(job_role.values):
    plt.text(
        i, v + 5, str(v), ha='center', fontweight='bold', fontsize=9)
plt.show()

 Q4. What is the average salary by job role?

In [ ]:
avg_sal = df.groupby('job_role')['salary'].mean().sort_values()
plt.figure(figsize=(10, 5))
avg_sal.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Average Salary by Job Role', fontsize=14, fontweight='bold')
plt.xlabel("Average Salary", fontsize=12)
plt.ylabel('Job Role', fontsize=12)
for i, v in enumerate(avg_sal):
    plt.text(v, i, f' {v:,.0f}', va='center')
plt.tight_layout()
plt.show()

Q5. What is the overall attrition distribution?

In [ ]:
# Pie chart — attrition proportion
att = df['attrition'].value_counts()

plt.figure(figsize=(5, 5))
att.plot(kind='pie', autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white'})
plt.title('Attrition Distribution', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.legend(
    ['Stayed (0)', 'Left (1)'],
    title='Attrition Status',
    loc='upper right'
)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — attrition count
att = df['attrition'].value_counts()

plt.figure(figsize=(6, 4))
att.plot(kind='bar', color=["#61BE62", "#DC6B6B"])
plt.title('Employee Attrition Count', fontsize=14, fontweight='bold')
plt.xlabel('Attrition')
plt.ylabel('Number of Employees')
for i, v in enumerate(att.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Insight:** Most employees remain with the company while a smaller portion has exited.

 Q6. How does attrition vary by job role?

In [ ]:
# Cross-tabulate attrition counts by job role
crosstab = pd.crosstab(df['job_role'], df['attrition'])
plt.figure(figsize=(6, 4))
crosstab.plot(kind='bar')
plt.title("Attrition by Job Role")
plt.xticks(rotation=45)
plt.show()

 Q7. What is the attrition trend over time?

In [ ]:
at['attrition_year'] = at['attrition_date'].dt.year
trend = at['attrition_year'].value_counts().sort_index()

plt.figure(figsize=(7, 4))
trend.plot(kind='line', linewidth=2, marker='o')
plt.title('Employee Attrition Trend by Year', fontsize=14, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Employees Left', fontsize=12)

for x, y in zip(trend.index, trend.values):
    plt.text(x, y + 1, str(y), ha='center', fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Insight:** Attrition increased over the years and peaked in recent periods.

 Q8. How many employees are in each department?

In [ ]:
# Merge employee and department tables on dept_id
emp_dpt = df.merge(dt, on='dept_id')
ed = emp_dpt['dept_name'].value_counts()
plt.figure(figsize=(7, 5))
ax = ed.plot(kind='bar', color="#77C7CB", edgecolor='black')
plt.title('Employee per Department')
plt.xlabel('Department', fontsize=12)
plt.ylabel('Number of Employee', fontsize=12)
plt.xticks(rotation=45)

for x, y in enumerate(ed.values):
    plt.text(x, y + 0.9, str(y), ha='center')
plt.tight_layout()
plt.show()

 Q9. Which department has the highest average salary?

In [ ]:
# Merge employee and department data, then compute average salary per department
df.merge(dt, on='dept_id')   .groupby('dept_name')['salary']   .mean()   .sort_values(ascending=False)

Q10. What is the distribution of performance ratings?

In [ ]:
perf = df['performance_rating'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
ax = perf.plot(kind='bar', edgecolor='black', color=["#ADFC3E"])
plt.title('Performance Rating Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Performance Rating')
plt.ylabel('Number of Employees')
plt.xticks(rotation=0)

# Add value labels on bars
for i, v in enumerate(perf.values):
    plt.text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

 Q11. Which department has the highest attrition?

In [ ]:
# Merge employee and department tables, then count attrition by department
df.merge(dt, on='dept_id')   .groupby('dept_name')['attrition']   .value_counts()

Q12. What is the distribution of attrition reasons?

In [ ]:
reason_count = at['attrition_reason'].value_counts().sort_values()

plt.figure(figsize=(8, 5))
ax = reason_count.plot(kind='barh', edgecolor='black', color=["#A23762"])
plt.title('Attrition Reason Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Employees')
plt.ylabel('Attrition Reason')

for i, v in enumerate(reason_count.values):
    plt.text(v + 1, i, str(v), va='center', fontweight='bold')

plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()